# 02 - dedupe and split

Takes metadata.csv from notebook 01 and builds train/val/test splits. Exact duplicates get dropped, near-duplicates stay but get grouped so a group never crosses a split. No image files are touched here, metadata only.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.width", 100)

Load the metadata table.

In [2]:
df = pd.read_csv("metadata.csv")
df["is_duplicate_of"] = df["is_duplicate_of"].fillna("")

print(df.shape)
df.head()

(1704, 8)


,filepath,class_label,width,height,filesize_bytes,format,phash,is_duplicate_of
0,C:\workstation\3RD YEAR\citrus_fruit\Citrus-Di...,aphids,4284,5712,1369832,JPEG,9996f6b9acd28d80,
1,C:\workstation\3RD YEAR\citrus_fruit\Citrus-Di...,aphids,3024,4032,1235446,JPEG,99b3f6b8ad528c84,
2,C:\workstation\3RD YEAR\citrus_fruit\Citrus-Di...,aphids,3024,4032,1234044,JPEG,99b5f6b1ad428ca4,
3,C:\workstation\3RD YEAR\citrus_fruit\Citrus-Di...,aphids,3024,4032,1263473,JPEG,99b7f6f18c408d85,
4,C:\workstation\3RD YEAR\citrus_fruit\Citrus-Di...,aphids,3024,4032,1281310,JPEG,9cb5f2f58c4a8ca4,


is_duplicate_of just points at another row, it doesn't say exact vs near. Compare phash strings to tell them apart - same hash means exact copy, different hash means near-duplicate.

In [3]:
phash_lookup = dict(zip(df["filepath"], df["phash"]))


def classify_duplicate(row):
    if row["is_duplicate_of"] == "":
        return "none"
    target_phash = phash_lookup[row["is_duplicate_of"]]
    return "exact" if row["phash"] == target_phash else "near"


df["duplicate_type"] = df.apply(classify_duplicate, axis=1)

counts = df["duplicate_type"].value_counts()
print(counts)

n_exact = int(counts.get("exact", 0))
n_near = int(counts.get("near", 0))
assert n_exact == 91, f"expected 91 exact duplicates, got {n_exact}"
assert n_near == 596, f"expected 596 near duplicates, got {n_near}"
print(f"\nmatches known facts: {n_exact} exact, {n_near} near")

duplicate_type
none     1017
near      596
exact      91
Name: count, dtype: int64

matches known facts: 91 exact, 596 near


Group images that are chained together through is_duplicate_of into one group_id, so a near-duplicate pair can never end up split across train and val. Plain union-find, nothing fancy needed for this size.

In [4]:
parent = {fp: fp for fp in df["filepath"]}


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]  # path compression
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb


for _, row in df.iterrows():
    if row["is_duplicate_of"] != "":
        union(row["filepath"], row["is_duplicate_of"])

roots = df["filepath"].map(find)
unique_roots = sorted(roots.unique())
root_to_group_id = {root: i for i, root in enumerate(unique_roots)}
df["group_id"] = roots.map(root_to_group_id)

group_sizes = df["group_id"].value_counts()
n_groups = df["group_id"].nunique()
n_multi_groups = int((group_sizes > 1).sum())

print(f"total groups: {n_groups}")
print(f"groups with more than one image: {n_multi_groups}")

total groups: 1017
groups with more than one image: 402


Drop the exact duplicates, they're redundant copies. Near-duplicates stay in, they still carry some visual variation.

In [5]:
n_before = len(df)

removed_exact = df[df["duplicate_type"] == "exact"][
    ["filepath", "class_label", "is_duplicate_of"]
].copy()

df = df[df["duplicate_type"] != "exact"].reset_index(drop=True)

n_after = len(df)
print(f"rows before: {n_before}")
print(f"rows after dropping exact duplicates: {n_after}")
assert n_after == 1613, f"expected 1613 rows remaining, got {n_after}"


rows before: 1704
rows after dropping exact duplicates: 1613


20-fold StratifiedGroupKFold keeps class balance while respecting groups, then just bucket the folds into train/val/test (14/3/3). Using the split() output directly for fold assignment rather than reshuffling anything by hand.

In [6]:
sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=42)

fold_of_row = np.full(len(df), -1, dtype=int)
for fold_idx, (_, test_idx) in enumerate(
    sgkf.split(df, df["class_label"], df["group_id"])
):
    fold_of_row[test_idx] = fold_idx

assert (fold_of_row >= 0).all(), "every row should have been assigned a fold"
df["fold"] = fold_of_row


def fold_to_split(fold_idx):
    if fold_idx <= 13:
        return "train"
    elif fold_idx <= 16:
        return "val"
    else:
        return "test"


df["split"] = df["fold"].map(fold_to_split)
df["split"].value_counts()

split
train    1131
val       242
test      240
Name: count, dtype: int64

Check counts, make sure no group got split across train/val/test, and confirm the exact duplicates are actually gone.

In [7]:
# 6a - counts per split and per class, with percentages
print("=== image counts per split ===")
split_counts = df["split"].value_counts()
for split_name in ["train", "val", "test"]:
    n = int(split_counts.get(split_name, 0))
    pct = 100 * n / len(df)
    print(f"{split_name:>5}: {n:5d}  ({pct:5.1f}%)")

print("\n=== per-class counts per split ===")
class_split_table = (
    df.groupby(["split", "class_label"]).size().unstack(fill_value=0)
)
class_split_pct = class_split_table.div(class_split_table.sum(axis=0), axis=1) * 100
print(class_split_table)
print()
print(class_split_pct.round(1))

=== image counts per split ===
train:  1131  ( 70.1%)
  val:   242  ( 15.0%)
 test:   240  ( 14.9%)

=== per-class counts per split ===
class_label  aphids  gummosis  healthy  leaf_minnor
split                                              
test             60        65       51           64
train           285       306      238          302
val              60        66       51           65

class_label  aphids  gummosis  healthy  leaf_minnor
split                                              
test           14.8      14.9     15.0         14.8
train          70.4      70.0     70.0         70.1
val            14.8      15.1     15.0         15.1


In [8]:
# 6b - a group_id should never span more than one split
splits_per_group = df.groupby("group_id")["split"].nunique()
violations = splits_per_group[splits_per_group > 1]

if len(violations) == 0:
    print("group integrity check passed: every group_id sits entirely within one split")
else:
    print(f"group integrity check FAILED: {len(violations)} group(s) span multiple splits")
    print(violations)

group integrity check passed: every group_id sits entirely within one split


In [9]:
# 6c - no exact duplicates should remain
remaining_exact = int((df["duplicate_type"] == "exact").sum())
print(f"remaining exact duplicates in working set: {remaining_exact}")
assert remaining_exact == 0

remaining exact duplicates in working set: 0


Write out the active split and a separate record of what got dropped.

In [10]:
output_cols = [
    "filepath",
    "class_label",
    "width",
    "height",
    "filesize_bytes",
    "format",
    "phash",
    "group_id",
    "duplicate_type",
    "split",
]

df[output_cols].to_csv("metadata_split.csv", index=False)
removed_exact.to_csv("removed_exact_duplicates.csv", index=False)

print("saved metadata_split.csv with", len(df), "rows")
print("saved removed_exact_duplicates.csv with", len(removed_exact), "rows")

saved metadata_split.csv with 1613 rows
saved removed_exact_duplicates.csv with 91 rows


Final summary.

In [11]:
print("Dataset split summary")
print("-" * 40)
print(f"images before dropping exact duplicates: {n_before}")
print(f"images after dropping exact duplicates:  {n_after}")
print(f"total duplicate groups:                  {n_groups}")
print(f"groups with more than one image:          {n_multi_groups}")
print()
print("per-split, per-class counts:")
print(class_split_table)

Dataset split summary
----------------------------------------
images before dropping exact duplicates: 1704
images after dropping exact duplicates:  1613
total duplicate groups:                  1017
groups with more than one image:          402

per-split, per-class counts:
class_label  aphids  gummosis  healthy  leaf_minnor
split                                              
test             60        65       51           64
train           285       306      238          302
val              60        66       51           65
